# Auditing an LLM judge panel with `metajudge`

An LLM-as-judge is a measurement instrument. Before its scores are trusted, two questions decide whether they mean anything:

1. **Reliability** — do independent judges agree? (Krippendorff's α, ICC)
2. **Differential item functioning (DIF)** — does the panel score one *kind* of output differently from another, conditional on quality?

Here three LLM judges score the **coherence (1–5)** of 16 short summaries, split by system family (**extractive** vs **abstractive**). `metajudge` then audits the panel.

> This notebook runs the **offline simulated** panel so it renders with no API key. Run `python examples/audit_llm_judge.py --mode live` for the real models.

In [ ]:
import sys
from pathlib import Path

here = Path.cwd()
ex_dir = here / "examples" if (here / "examples").exists() else here
sys.path.insert(0, str(ex_dir))

import pandas as pd  # noqa: E402
from audit_llm_judge import load_items, score_offline  # noqa: E402

items = load_items()
pd.DataFrame(items)[["id", "system_family", "summary"]]

## The judge panel

Three distinct models (not one model at temperature > 0) act as independent judges — distinct models is what makes "do the judges agree?" a real question. Each scores every summary 1–5 on coherence. The simulated panel below is seeded and deterministic; live mode replaces it with real model calls.

In [2]:
scores = score_offline(items)

# the score matrix: 16 items (rows) x 3 judges (columns)
scores.pivot(index="item", columns="rater", values="score")

rater,gemini-2.5-flash,gemini-3-flash,gemini-3.5-flash
item,,,
doc01,5,4,4
doc02,5,4,4
doc03,5,4,5
doc04,4,4,5
doc05,4,3,4
doc06,5,4,5
doc07,4,5,5
doc08,5,5,5
doc09,3,1,3


## The pattern, at a glance

Mean coherence by judge and system family. Two things to look for: judges sitting at different heights (leniency differences → a reliability question), and abstractive bars below extractive (a possible DIF signal).

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

means = scores.pivot_table(index="rater", columns="system_family", values="score", aggfunc="mean")
ax = means.plot(kind="bar", figsize=(7, 4), color=["#C44E52", "#4C72B0"], edgecolor="white")
ax.set_ylim(1, 5)
ax.set_ylabel("mean coherence (1-5)")
ax.set_xlabel("judge model")
ax.set_title("Mean coherence by judge and system family")
ax.legend(title="system family")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## The audit

`audit()` returns a one-screen report card: reliability (α, ICC) and DIF across the abstractive-vs-extractive stratum.

In [4]:
from IPython.display import Markdown

from metajudge import audit, cluster_bootstrap_dif
from metajudge.data import Ratings

ratings = Ratings.from_long(
    scores, item="item", rater="rater", score="score", stratum="system_family"
)
card = audit(ratings, focal="abstractive", reference="extractive")
Markdown(card.to_markdown())

/Users/Apple/Developer/portfolio_llm/judge-audit/src/metajudge/report.py:150: UserWarning: n_obs=48: the Jodoin-Gierl A/B/C classification was calibrated on educational testing datasets with ≥500 examinees; at this sample size the R²-change thresholds overlap and the class should be treated as indicative only.
  dif=logistic_dif(ratings, focal=focal, reference=reference, conditioner=conditioner),


# metajudge report card

## Reliability
- Krippendorff's alpha (ordinal): 0.824 [95% CI 0.617, 0.889]
- ICC(2,1): 0.835; ICC(2,k): 0.938 (16 targets x 3 raters)

## DIF (panel-relative, rest-score conditioner)
> Note: the rest-score conditioner cannot see bias shared across the entire rater panel, so this is panel-relative DIF, not an instrument-level fairness clearance. Pass a valid independent external quality conditioner for a stronger instrument-level analysis.

- abstractive vs extractive (conditioner: rest_score, n=48)
- Uniform DIF: chi2(1)=8.69, p=0.0032
- Nonuniform DIF: chi2(1)=0.87, p=0.3497
- Effect size (Nagelkerke R2 delta): 0.058 (Jodoin-Gierl class B)

In [5]:
cb = cluster_bootstrap_dif(ratings, focal="abstractive", reference="extractive", n_boot=200, seed=0)
print(
    f"Nagelkerke R² delta: {cb.base.nagelkerke_r2_delta:.3f} "
    f"[95% cluster CI {cb.r2_delta_ci_low:.3f}, {cb.r2_delta_ci_high:.3f}]"
)
print(f"CI reliable: {cb.ci_reliable} (n_effective={cb.n_effective} of {cb.n_boot})")

/Users/Apple/Developer/portfolio_llm/judge-audit/src/metajudge/dif.py:656: UserWarning: n_obs=48: the Jodoin-Gierl A/B/C classification was calibrated on educational testing datasets with ≥500 examinees; at this sample size the R²-change thresholds overlap and the class should be treated as indicative only.
  base = logistic_dif(ratings, focal=focal, reference=reference, conditioner=conditioner)


Nagelkerke R² delta: 0.058 [95% cluster CI 0.023, 0.244]
CI reliable: True (n_effective=199 of 200)


## How to read it

- **Reliability** — α ≥ 0.667 is tentatively reliable, ≥ 0.80 reliable; ICC(2,k) averages out per-judge noise.
- **DIF** — a significant uniform-DIF test with a *small* effect size means the difference is detectable but minor; a class B/C effect means the panel is materially scoring abstractive outputs differently once quality is matched — a reason to revise the rubric or the judges before trusting the scores.

The numbers above are from the **seeded simulation**. For a real audit:

```bash
pip install metajudge[examples]   # or: uv add --optional examples google-genai
export GOOGLE_AI_API_KEY=...
python examples/audit_llm_judge.py --mode live
```